# Hosting Strands Agents with Amazon Bedrock models in Amazon Bedrock AgentCore Runtime

## Overview

In this tutorial we will learn how to host your existing agent, using Amazon Bedrock AgentCore Runtime. We will provide examples using Amazon Bedrock models and non-Bedrock models such as Azure OpenAI and Gemini.


### Tutorial Details


| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Conversational                                                                   |
| Agent type          | Single                                                                           |
| Agentic Framework   | Strands Agents                                                                   |
| LLM model           | Anthropic Claude Haiku 4.5                                                        |
| Tutorial components | Hosting agent on AgentCore Runtime. Using Strands Agent and Amazon Bedrock Model |
| Tutorial vertical   | Cross-vertical                                                                   |
| Example complexity  | Easy                                                                             |
| SDK used            | Amazon BedrockAgentCore Python SDK and boto3                                     |

### Tutorial Architecture

In this tutorial we will describe how to deploy an existing agent to AgentCore runtime. 

For demonstration purposes, we will  use a Strands Agent using Amazon Bedrock models

In our example we will use a very simple agent with two tools: `get_weather` and `get_time`. 

<div style="text-align:left">
    <img src="images/architecture_runtime.png" width="50%"/>
</div>

### Tutorial Key Features

* Hosting Agents on Amazon Bedrock AgentCore Runtime
* Using Amazon Bedrock models
* Using Strands Agents


## Prerequisites

To execute this tutorial you will need:
* Python 3.10+
* AWS credentials
* Amazon Bedrock AgentCore SDK
* Strands Agents

In [1]:
!pip install --force-reinstall -U -r requirements.txt --quiet


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Creating your agents and experimenting locally

Before we deploy our agents to AgentCore Runtime, let's develop and run them locally for experimentation purposes.

For production agentic applications we will need to decouple the agent creation process from the agent invocation one. With AgentCore Runtime, we will decorate the invocation part of our agent with the `@app.entrypoint` decorator and have it as the entry point for our runtime. Let's first look how each agent is developed during the experimentation phase.

The architecture here will look as following:

<div style="text-align:left">
    <img src="images/architecture_local.png" width="50%"/>
</div>

In [2]:
%%writefile strands_lgpd.py
import argparse
import json
import logging
import re
from datetime import datetime, timezone

from strands import Agent, tool
from strands.models import BedrockModel

logger = logging.getLogger("holocron_sentinel")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s - %(message)s")


@tool
def validar_termo_consentimento(texto: str) -> str:
    """Valida rapidamente presença de itens essenciais de consentimento LGPD."""
    t = (texto or "").lower()
    checks = {
        "finalidade": any(k in t for k in ["finalidade", "propósito", "objetivo"]),
        "base_legal": any(k in t for k in ["base legal", "consentimento", "art.", "artigo"]),
        "direitos_titular": any(k in t for k in ["direitos", "revogação", "acesso", "correção", "portabilidade", "eliminação"]),
        "canal_dpo": any(k in t for k in ["encarregado", "dpo", "contato", "canal"]),
        "retencao": any(k in t for k in ["retenção", "prazo", "armazenamento"]),
    }
    faltando = [k for k, ok in checks.items() if not ok]
    if not faltando:
        return "OK: termo parece conter itens mínimos (finalidade, base legal, direitos, canal DPO, retenção)."
    return "PENDENTE: faltando itens no termo: " + ", ".join(faltando)


@tool
def anonimizar_texto(texto: str) -> str:
    """Anonimiza padrões comuns (CPF, e-mail) em texto livre."""
    if not texto:
        return ""
    out = re.sub(r"\b\d{3}\.\d{3}\.\d{3}-\d{2}\b", "***.***.***-**", texto)
    out = re.sub(r"[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}", "***@***.***", out, flags=re.IGNORECASE)
    return out


@tool
def registrar_evento_auditoria(evento_json: str) -> str:
    """Registra (simulado) um evento de auditoria de compliance."""
    try:
        evento = json.loads(evento_json) if isinstance(evento_json, str) else evento_json
    except Exception:
        evento = {"raw": str(evento_json)}
    envelope = {
        "ts": datetime.now(timezone.utc).isoformat(),
        "tipo": "AUDITORIA_COMPLIANCE",
        "evento": evento,
    }
    logger.info("AUDIT_EVENT %s", json.dumps(envelope, ensure_ascii=False))
    return "AUDIT_OK"


model_id = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
model = BedrockModel(model_id=model_id)

agent = Agent(
    model=model,
    tools=[validar_termo_consentimento, anonimizar_texto, registrar_evento_auditoria],
    system_prompt=(
        "Você é o Holocron Sentinel V2, um agente corporativo de Compliance LGPD. "
        "Ajude a validar termos de consentimento, sugerir ajustes e orientar auditoria. "
        "Quando houver dados pessoais no texto, prefira anonimizar antes de registrar eventos."
    ),
)


def strands_agent_bedrock(payload):
    user_input = (payload or {}).get("prompt", "")
    response = agent(user_input)
    return response.message["content"][0]["text"]


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("payload", type=str)
    args = parser.parse_args()
    print(strands_agent_bedrock(json.loads(args.payload)))

Overwriting strands_lgpd.py


#### Invoking local agent

In [3]:
!python strands_lgpd.py '{"prompt": "Valide este termo de consentimento (responda com itens faltantes e melhorias): \"Coletamos dados para melhorar o serviço. Você pode revogar quando quiser. Contato: dpo@empresa.com\""}'

2026-03-20 10:42:00,475 INFO botocore.credentials - Found credentials in shared credentials file: ~/.aws/credentials
usage: strands_lgpd.py [-h] payload
strands_lgpd.py: error: unrecognized arguments: Valide este termo de consentimento (responda com itens faltantes e melhorias): "Coletamos dados para melhorar o serviço. Você pode revogar quando quiser. Contato: dpo@empresa.com"}'


## Preparing your agent for deployment on AgentCore Runtime

Let's now deploy our agents to AgentCore Runtime. To do so we need to:
* Import the Runtime App with `from bedrock_agentcore.runtime import BedrockAgentCoreApp`
* Initialize the App in our code with `app = BedrockAgentCoreApp()`
* Decorate the invocation function with the `@app.entrypoint` decorator
* Let AgentCoreRuntime control the running of the agent with `app.run()`

### Strands Agents with Amazon Bedrock model
Let's start with our Strands Agent using Amazon Bedrock model. All the others will work exactly the same.

In [4]:
%%writefile strands_lgpd.py
import json
import logging
import re
from datetime import datetime, timezone

from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent, tool
from strands.models import BedrockModel

logger = logging.getLogger("holocron_sentinel")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s - %(message)s")

app = BedrockAgentCoreApp()


@tool
def validar_termo_consentimento(texto: str) -> str:
    t = (texto or "").lower()
    checks = {
        "finalidade": any(k in t for k in ["finalidade", "propósito", "objetivo"]),
        "base_legal": any(k in t for k in ["base legal", "consentimento", "art.", "artigo"]),
        "direitos_titular": any(k in t for k in ["direitos", "revogação", "acesso", "correção", "portabilidade", "eliminação"]),
        "canal_dpo": any(k in t for k in ["encarregado", "dpo", "contato", "canal"]),
        "retencao": any(k in t for k in ["retenção", "prazo", "armazenamento"]),
    }
    faltando = [k for k, ok in checks.items() if not ok]
    if not faltando:
        return "OK: termo parece conter itens mínimos (finalidade, base legal, direitos, canal DPO, retenção)."
    return "PENDENTE: faltando itens no termo: " + ", ".join(faltando)


@tool
def anonimizar_texto(texto: str) -> str:
    if not texto:
        return ""
    out = re.sub(r"\b\d{3}\.\d{3}\.\d{3}-\d{2}\b", "***.***.***-**", texto)
    out = re.sub(r"[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}", "***@***.***", out, flags=re.IGNORECASE)
    return out


@tool
def registrar_evento_auditoria(evento_json: str) -> str:
    try:
        evento = json.loads(evento_json) if isinstance(evento_json, str) else evento_json
    except Exception:
        evento = {"raw": str(evento_json)}
    envelope = {
        "ts": datetime.now(timezone.utc).isoformat(),
        "tipo": "AUDITORIA_COMPLIANCE",
        "evento": evento,
    }
    logger.info("AUDIT_EVENT %s", json.dumps(envelope, ensure_ascii=False))
    return "AUDIT_OK"


model_id = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
model = BedrockModel(model_id=model_id)

agent = Agent(
    model=model,
    tools=[validar_termo_consentimento, anonimizar_texto, registrar_evento_auditoria],
    system_prompt=(
        "Você é o Holocron Sentinel V2, um agente corporativo de Compliance LGPD. "
        "Ajude a validar termos de consentimento, sugerir ajustes e orientar auditoria. "
        "Quando houver dados pessoais no texto, prefira anonimizar antes de registrar eventos."
    ),
)


@app.entrypoint
def strands_agent_bedrock(payload):
    user_input = (payload or {}).get("prompt", "")
    logger.info("INVOKE prompt_len=%s", len(user_input))
    response = agent(user_input)
    return response.message["content"][0]["text"]


if __name__ == "__main__":
    app.run()

Overwriting strands_lgpd.py


## What happens behind the scenes?

When you use `BedrockAgentCoreApp`, it automatically:

* Creates an HTTP server that listens on the port 8080
* Implements the required `/invocations` endpoint for processing the agent's requirements
* Implements the `/ping` endpoint for health checks (very important for asynchronous agents)
* Handles proper content types and response formats
* Manages error handling according to the AWS standards

## Deploying the agent to AgentCore Runtime

The `CreateAgentRuntime` operation supports comprehensive configuration options, letting you specify container images, environment variables and encryption settings. You can also configure protocol settings (HTTP, MCP) and authorization mechanisms to control how your clients communicate with the agent. 

**Note:** Operations best practice is to package code as container and push to ECR using CI/CD pipelines and IaC

In this tutorial can will the Amazon Bedrock AgentCore Python SDK to easily package your artifacts and deploy them to AgentCore runtime.

### Configure AgentCore Runtime deployment

First we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch.

During the configure step, your docker file will be generated based on your application code

<div style="text-align:left">
    <img src="images/configure.png" width="60%"/>
</div>

In [5]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "holocron_sentinel_lgpd_getting_started"
response = agentcore_runtime.configure(
    entrypoint="strands_lgpd.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name
)
response

c:\Users\barre\AWS-reStart-Compliance-Portfolio\AWS-re-Start\P - Holocron-Sentinel\venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
Entrypoint parsed: file=C:\Users\barre\AWS-reStart-Compliance-Portfolio\AWS-re-Start\P - Holocron-Sentinel\amazon-bedrock-agentcore-samples\01-tutorials\01-AgentCore-runtime\01-hosting-agent\01-strands-with-bedrock-model\strands_lgpd.py, bedrock_agentcore_name=strands_lgpd
Configuring BedrockAgentCore agent: holocron_sentinel_lgpd_getting_started


⚠️  ℹ️  No container engine found (Docker/Finch/Podman not installed)
✅ Default deployment uses CodeBuild (no container engine needed)
💡 Run 'agentcore launch' for cloud-based building and deployment
💡 For local builds, install Docker, Finch, or Podman

⚠️  [WARNING] Platform mismatch: Current system is 'linux/amd64' but Bedrock AgentCore requires 'linux/arm64'.
For deployment options and workarounds, see: 
https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html

Generated Dockerfile: c:\Users\barre\AWS-reStart-Compliance-Portfolio\AWS-re-Start\P - Holocron-Sentinel\amazon-bedrock-agentcore-samples\01-tutorials\01-AgentCore-runtime\01-hosting-agent\01-strands-with-bedrock-model\Dockerfile
Generated .dockerignore: c:\Users\barre\AWS-reStart-Compliance-Portfolio\AWS-re-Start\P - Holocron-Sentinel\amazon-bedrock-agentcore-samples\01-tutorials\01-AgentCore-runtime\01-hosting-agent\01-strands-with-bedrock-model\.dockerignore
Keeping 'holocron_sentinel_lgpd_getting_started' as default agent
Bedrock AgentCore configured: c:\Users\barre\AWS-reStart-Compliance-Portfolio\AWS-re-Start\P - Holocron-Sentinel\amazon-bedrock-agentcore-samples\01-tutorials\01-AgentCore-runtime\01-hosting-agent\01-strands-with-bedrock-model\.bedrock_agentcore.yaml


ConfigureResult(config_path=WindowsPath('c:/Users/barre/AWS-reStart-Compliance-Portfolio/AWS-re-Start/P - Holocron-Sentinel/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/01-hosting-agent/01-strands-with-bedrock-model/.bedrock_agentcore.yaml'), dockerfile_path=WindowsPath('c:/Users/barre/AWS-reStart-Compliance-Portfolio/AWS-re-Start/P - Holocron-Sentinel/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/01-hosting-agent/01-strands-with-bedrock-model/Dockerfile'), dockerignore_path=WindowsPath('c:/Users/barre/AWS-reStart-Compliance-Portfolio/AWS-re-Start/P - Holocron-Sentinel/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/01-hosting-agent/01-strands-with-bedrock-model/.dockerignore'), runtime='None', region='us-east-1', account_id='050002895582', execution_role=None, ecr_repository=None, auto_create_ecr=True)

### Launching agent to AgentCore Runtime

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime

<div style="text-align:left">
    <img src="images/launch.png" width="75%"/>
</div>

In [6]:
launch_result = agentcore_runtime.launch()

🚀 CodeBuild mode: building in cloud (RECOMMENDED - DEFAULT)
   • Build ARM64 containers in the cloud with CodeBuild
   • No local Docker required
💡 Available deployment modes:
   • runtime.launch()                           → CodeBuild (current)
   • runtime.launch(local=True)                 → Local development
   • runtime.launch(local_build=True)           → Local build + cloud deploy (NEW)
Starting CodeBuild ARM64 deployment for agent 'holocron_sentinel_lgpd_getting_started' to account 050002895582 (us-east-1)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: holocron_sentinel_lgpd_getting_started
✅ ECR repository available: 050002895582.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-holocron_sentinel_lgpd_getting_started
Getting or creating execution role for agent: holocron_sentinel_lgpd_getting_started
Using AWS region: us-east-1, account ID: 050002895582
Role name: AmazonBedrockAgentCoreSDKRuntime-us-east-1-ebe47c50ad

✅ Reusing existing ECR repository: 050002895582.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-holocron_sentinel_lgpd_getting_started


✅ Reusing existing execution role: arn:aws:iam::050002895582:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-ebe47c50ad
✅ Execution role available: arn:aws:iam::050002895582:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-ebe47c50ad
Preparing CodeBuild project and uploading source...
Getting or creating CodeBuild execution role for agent: holocron_sentinel_lgpd_getting_started
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-ebe47c50ad
Reusing existing CodeBuild execution role: arn:aws:iam::050002895582:role/AmazonBedrockAgentCoreSDKCodeBuild-us-east-1-ebe47c50ad
Using .dockerignore with 44 patterns
Uploaded source to S3: holocron_sentinel_lgpd_getting_started/source.zip
Updated CodeBuild project: bedrock-agentcore-holocron_sentinel_lgpd_getting_started-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.2s
🔄 PROVISIONING started (total: 1s)
✅ PROVISIONING completed in 7.1s
🔄 DO

### Checking for the AgentCore Runtime Status
Now that we've deployed the AgentCore Runtime, let's check for it's deployment status

In [8]:
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

Retrieved Bedrock AgentCore status for: holocron_sentinel_lgpd_getting_started


'READY'

### Invoking AgentCore Runtime

Finally, we can invoke our AgentCore Runtime with a payload

<div style="text-align:left">
    <img src="images/invoke.png" width=75%"/>
</div>

In [10]:
({"prompt": "How is the weather now?"})
invoke_response = agentcore_runtime.invoke({"prompt": "Valide rapidamente um termo de consentimento LGPD."})

### Processing invocation results

We can now process our invocation results to include it in an application

In [11]:
from IPython.display import Markdown, display
import json
response_text = invoke_response['response'][0]
display(Markdown(response_text))

Eu estou pronto para validar um termo de consentimento LGPD! 

Para fazer isso, preciso que você me **compartilhe o texto do termo** que deseja validar.

Por favor, forneça:
- O conteúdo completo do termo de consentimento, ou
- A parte específica que gostaria de revisar

Após receber o texto, farei uma validação completa verificando:
✓ Conformidade com a Lei Geral de Proteção de Dados (LGPD)
✓ Clareza e transparência das informações
✓ Definição adequada de finalidades
✓ Consentimento livre e informado
✓ Direitos do titular dos dados
✓ Pontos de melhoria

Qual é o termo que você gostaria de validar?

### Invoking AgentCore Runtime with boto3

Now that your AgentCore Runtime was created you can invoke it with any AWS SDK. For instance, you can use the boto3 `invoke_agent_runtime` method for it.

In [ ]:
import boto3
agent_arn = launch_result.agent_arn
agentcore_client = boto3.client(
    'bedrock-agentcore',
    region_name=region
)

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "What is 2+2?"})
)

# Capture the runtime session ID for lifecycle management
runtime_session_id = boto3_response.get('runtimeSessionId')
print(f"Runtime Session ID: {runtime_session_id}")

if "text/event-stream" in boto3_response.get("contentType", ""):
    content = []
    for line in boto3_response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                line = line[6:]
                print(line)
                content.append(line)
    display(Markdown("\n".join(content)))
else:
    try:
        events = []
        for event in boto3_response.get("response", []):
            events.append(event)
    except Exception as e:
        events = [f"Error reading EventStream: {e}"]
    display(Markdown(json.loads(events[0].decode("utf-8"))))

### Stopping a Session

You'll want to stop individual sessions when they're no longer needed.
This releases the microVM resources for that session while keeping the runtime alive
for new sessions. Below we demonstrate `stop_runtime_session`.

In [ ]:
# --- Inline Session Lifecycle Demo ---
# stop_runtime_session releases the microVM resources for this specific session while keeping the runtime alive for new sessions.


if runtime_session_id:
    agentcore_client.stop_runtime_session(
        agentRuntimeArn=agent_arn,
        runtimeSessionId=runtime_session_id,
        qualifier='DEFAULT'
    )
    print(f"✅ Session '{runtime_session_id}' stopped — microVM resources released")
else:
    print("⚠️ No session ID available to stop")

### Lifecycle Configuration Demo
Now let's demonstrate how to configure a runtime with a shorter idle timeout.
We'll create a second runtime with a 5-minute (300 second) idle timeout to show
how lifecycle configuration affects session behavior. Both runtimes will coexist.

In [ ]:
# --- Lifecycle Configuration Demo ---
# In production, choose a timeout appropriate for your workload:
#   - Development/testing: 5-15 minutes
#   - Interactive sessions: 30-60 minutes
#   - Long-running workloads: adjust as needed
#

agentcore_runtime_short = Runtime()
agent_name_short = "holocron_sentinel_lgpd_short_timeout"

# Configure with shorter idle timeout
response_short = agentcore_runtime_short.configure(
    entrypoint="strands_lgpd.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name_short
)

# Launch the second runtime
launch_result_short = agentcore_runtime_short.launch()
print(f"Second runtime launched: {launch_result_short.agent_id}")

# Wait for it to be ready
status_response_short = agentcore_runtime_short.status()
status_short = status_response_short.endpoint['status']
while status_short not in ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']:
    time.sleep(10)
    status_response_short = agentcore_runtime_short.status()
    status_short = status_response_short.endpoint['status']
    print(f"Short timeout runtime status: {status_short}")

# Now update the runtime with shorter idle timeout using boto3
# UpdateAgentRuntime is a full-replacement API — we must re-supply all required fields.
# First, retrieve the current runtime configuration.
agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=region)
current_runtime = agentcore_control_client.get_agent_runtime(
    agentRuntimeId=launch_result_short.agent_id
)

update_response = agentcore_control_client.update_agent_runtime(
    agentRuntimeId=launch_result_short.agent_id,
    agentRuntimeArtifact=current_runtime['agentRuntimeArtifact'],
    roleArn=current_runtime['roleArn'],
    networkConfiguration=current_runtime['networkConfiguration'],
    lifecycleConfiguration={
        'idleRuntimeSessionTimeout': 300  # 5 minutes
    }
)
print(f"✅ Runtime updated with 5-minute idle timeout")

# Invoke the second runtime to verify it works
invoke_response_short = agentcore_runtime_short.invoke({"prompt": "What is 3+3?"})
print(f"Second runtime response: {invoke_response_short['response'][0]}")

## Session Lifecycle Best Practices

AgentCore Runtime costs are based on vCPU and Memory. A best practice to avoid undesired costs is to explicitly stop the session or set up a properly configured idle timeout, so the session will be terminated.

To manage costs effectively:

- **Configure idle timeout**: Set an appropriate idle timeout during session creation to automatically stop inactive sessions. Choose a value based on your use case (e.g., shorter for development/testing, longer for production workloads).
- **Stop sessions when done**: Use `stop_runtime_session` to release the microVM resources for a specific session while keeping the runtime alive for new sessions.

## Cleanup

Let's now clean up the AgentCore Runtime and associated resources. We delete the runtime first to avoid undesired costs, then clean up supporting resources like ECR repositories.

In [ ]:
launch_result.ecr_uri, launch_result.agent_id, launch_result.ecr_uri.split('/')[1]

In [ ]:
# --- Stop active sessions to release microVM resources ---
import boto3

agentcore_client = boto3.client('bedrock-agentcore', region_name=region)
agentcore_control_client = boto3.client('bedrock-agentcore-control', region_name=region)
ecr_client = boto3.client('ecr', region_name=region)

# Stop the active session to release its microVM resources
# In production, this is how you end individual user sessions while keeping the runtime alive
# AgentCore Runtime costs are based on vCPU and Memory — stopping sessions avoids undesired costs
# Note: If the session was already stopped in the earlier demo cell, this will raise a
# ResourceNotFoundException — the except block handles that gracefully.
if 'runtime_session_id' in locals() and runtime_session_id:
    try:
        agentcore_client.stop_runtime_session(
            agentRuntimeArn=launch_result.agent_arn,
            runtimeSessionId=runtime_session_id,
            qualifier='DEFAULT'
        )
        print(f"✅ Session '{runtime_session_id}' stopped")
    except Exception as e:
        print(f"⚠️ Failed to stop session '{runtime_session_id}': {e}")

# --- Delete both runtimes ---
# Original runtime
try:
    agentcore_control_client.delete_agent_runtime(
        agentRuntimeId=launch_result.agent_id,
    )
    print(f"✅ Original runtime '{launch_result.agent_id}' deleted")
except Exception as e:
    print(f"⚠️ Failed to delete original runtime: {e}")

# Short-timeout runtime
if 'launch_result_short' in locals():
    try:
        agentcore_control_client.delete_agent_runtime(
            agentRuntimeId=launch_result_short.agent_id,
        )
        print(f"✅ Short-timeout runtime '{launch_result_short.agent_id}' deleted")
    except Exception as e:
        print(f"⚠️ Failed to delete short-timeout runtime: {e}")

# --- Delete ECR repositories ---
try:
    ecr_client.delete_repository(
        repositoryName=launch_result.ecr_uri.split('/')[1],
        force=True
    )
    print(f"✅ ECR repository '{launch_result.ecr_uri.split('/')[1]}' deleted")
except Exception as e:
    print(f"⚠️ Failed to delete ECR repository: {e}")

if 'launch_result_short' in locals():
    try:
        ecr_client.delete_repository(
            repositoryName=launch_result_short.ecr_uri.split('/')[1],
            force=True
        )
        print(f"✅ Second ECR repository '{launch_result_short.ecr_uri.split('/')[1]}' deleted")
    except Exception as e:
        print(f"⚠️ Failed to delete second ECR repository: {e}")

# Congratulations!